# 🧩 Heurística Shelf First Fit para o Problema de Empacotamento 2D

Este projeto tem como objetivo desenvolver e implementar uma **heurística de construção Shelf First Fit** para resolver o **Problema de Empacotamento Bidimensional (2D Bin Packing Problem)**, amplamente estudado na área de otimização e logística.

## 📦 Descrição do Problema

Dado um conjunto de itens retangulares, cada um com largura e altura conhecidas, o desafio consiste em empacotá-los dentro de um recipiente bidimensional de dimensões fixas, **sem sobreposição**, buscando **maximizar o aproveitamento do espaço disponível**.

O problema pertence à classe dos **NP-difíceis**, tornando heurísticas construtivas uma alternativa eficiente para gerar boas soluções em tempo computacional reduzido.

## 📚 Heurística Shelf First Fit

A heurística **Shelf First Fit** organiza o empacotamento da seguinte forma:

- O recipiente é dividido em **prateleiras (shelves)** horizontais  
- Cada item é colocado na **primeira prateleira disponível** onde ele cabe  
- Caso o item não caiba em nenhuma prateleira existente, uma **nova prateleira é criada**  
- A altura da prateleira é definida pelo **item mais alto nela alocado**

---



In [1]:
from google.colab import files
uploaded = files.upload()

Saving exemplo_muitos_itens.csv to exemplo_muitos_itens.csv
Saving exemplo_variado.csv to exemplo_variado.csv
Saving exemplo_pequeno.csv to exemplo_pequeno.csv


###**Importação das bibliotecas**

In [2]:
import csv
import copy
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.colors as mcolors
import os

# interactive visualization
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown, Button, VBox, HBox, Output, Label

###**Definição das classes:**

In [3]:
class Item:
   def __init__(self, width, height, id=None):
        self.width = width    # Largura do item
        self.height = height  # Altura do item
        self.id = id          # Identificador único
        self.x = None         # Posição X final no Bin
        self.y = None         # Posição Y final no Bin
        self.shelf = None     # Referência à prateleira onde foi colocado

In [4]:
class Shelf:
    def __init__(self, y_position, max_width, height):
        self.y = y_position       # Posição vertical da base da prateleira
        self.max_width = max_width # Largura total disponível (largura do Bin)
        self.height = height      # Altura da prateleira (definida pelo 1º item)
        self.used_width = 0       # Largura já ocupada por itens nesta prateleira
        self.items = []           # Lista de itens alocados aqui

    def can_place(self, item):
        """Verifica se o item cabe na largura restante e se não é mais alto que a prateleira."""
        return (self.used_width + item.width <= self.max_width and
                item.height <= self.height)

    def place_item(self, item):
        """Posiciona o item na prateleira e atualiza o espaço usado."""
        item.x = self.used_width  # O item começa onde o espaço usado termina
        item.y = self.y           # A base do item é a base da prateleira
        item.shelf = self
        self.items.append(item)
        self.used_width += item.width


In [5]:
class Bin:
    def __init__(self, width, height):
        self.width = width           # Largura total do contêiner
        self.height = height         # Altura total do contêiner
        self.shelves = []            # Lista de prateleiras criadas
        self.current_height = 0      # Altura total ocupada pelas prateleiras

    def recompute_positions(self):
          """
          Recalcula x, y e shelf de todos os itens do bin.
          """
          for shelf in self.shelves:
              x_cursor = 0.0
              for item in shelf.items:
                  item.x = x_cursor
                  item.y = shelf.y
                  item.shelf = shelf
                  x_cursor += item.width

###**Carregamento de itens:**

In [6]:
def load_items_from_csv(file_name):
    items = []
    if not os.path.exists(file_name):
        return []
    with open(file_name, newline="", encoding="utf-8") as csvfile:
        reader = csv.DictReader(csvfile)
        for i, row in enumerate(reader, start=1):
            width = row.get("width") or row.get("largura") or row.get("w") or row.get("W")
            height = row.get("height") or row.get("altura") or row.get("h") or row.get("H")
            if width and height:
                items.append(Item(width=float(width), height=float(height), id=i))
    return items

| **Situação**                              | **Comportamento do algoritmo**                         |
|-------------------------------------|----------------------------------------------------|
| `alpha = 0`                          | Guloso (Shelf First-Fit clássico); `seed` não tem efeito |
| `alpha > 0` e `seed = 42`            | Pseudo-aleatório reprodutível (mesma solução sempre) |
| `alpha > 0` e `seed = None`          | Pseudo-aleatório não reprodutível (solução diferente a cada execução) |

In [7]:
def shelf_first_fit_gasp(items, bin_width, bin_height, alpha=0.0, seed=None):
    rng = random.Random(seed)
    bin_obj = Bin(bin_width, bin_height)

    # Histórico para o Slider (passo a passo)
    history = [copy.deepcopy(bin_obj)]
    decisions = ["Início: Bin vazio"]

    for step, item in enumerate(items, start=1):
        # 1. Identificar candidatos (Prateleiras existentes onde o item cabe)
        candidates = [{"type": "existing", "obj": s, "idx": i}
                      for i, s in enumerate(bin_obj.shelves) if s.can_place(item)]

        # 2. Considerar criar uma nova prateleira como um candidato
        if bin_obj.current_height + item.height <= bin_obj.height:
            candidates.append({"type": "new"})

        # Função de custo: quanto espaço sobrará na prateleira após colocar o item?
        def get_cost(c):
            if c["type"] == "existing":
                return c["obj"].max_width - (c["obj"].used_width + item.width)
            return bin_obj.width - item.width

        # Ordenar candidatos do melhor (menor sobra) para o pior
        ordered = sorted(candidates, key=get_cost)

        if not ordered:
            decisions.append(f"Item {item.id}: DESCARTADO (Sem espaço vertical ou horizontal)")
        else:
            # O alpha define o tamanho da lista de melhores candidatos.
            # Se alpha=0, limit=1 (pega sempre o melhor).
            # Se alpha=1, limit=total (escolhe qualquer um que caiba).
            limit = max(1, round(len(ordered) * alpha))
            if alpha > 0 and len(ordered) > 1:
                limit = max(limit, 2)
            limit = min(limit, len(ordered))

            rcl = ordered[:limit]      # Lista restrita dos melhores
            chosen = rng.choice(rcl)   # Escolha aleatória dentro da lista restrita

            if chosen["type"] == "existing":
                s = chosen["obj"]
                s.place_item(item)
                decisions.append(f"Item {item.id} -> Shelf {chosen['idx']}")
            else:
                # Criar nova prateleira com a altura do item atual
                new_shelf = Shelf(y_position=bin_obj.current_height,
                                  max_width=bin_obj.width,
                                  height=item.height)
                new_shelf.place_item(item)
                bin_obj.shelves.append(new_shelf)
                bin_obj.current_height += new_shelf.height
                decisions.append(f"Item {item.id} -> Nova Shelf {len(bin_obj.shelves)-1} ")

        # Salva o estado atual para a visualização
        history.append(copy.deepcopy(bin_obj))

    return history, decisions

**Visualização:**

In [8]:
def plot_bin_step(bin_obj, title=""):
    fig, ax = plt.subplots(figsize=(7, 8))
    ax.add_patch(patches.Rectangle((0, 0), bin_obj.width, bin_obj.height, linewidth=2, edgecolor='black', facecolor='#f0f0f0'))

    colors = list(mcolors.TABLEAU_COLORS.values())

    for i, shelf in enumerate(bin_obj.shelves):
        shelf_color = colors[i % len(colors)]
        # Área da prateleira
        ax.add_patch(patches.Rectangle((0, shelf.y), bin_obj.width, shelf.height, facecolor='white', edgecolor='gray', linestyle='--', alpha=0.3))

        for item in shelf.items:
            ax.add_patch(patches.Rectangle((item.x, item.y), item.width, item.height,
                                         facecolor=shelf_color, edgecolor='black', alpha=0.8))
            ax.text(item.x + item.width/2, item.y + item.height/2, f"{int(item.id)}",
                    ha='center', va='center', fontsize=9, color='white', weight='bold')

    ax.set_xlim(-0.5, bin_obj.width + 0.5)
    ax.set_ylim(-0.5, bin_obj.height + 0.5)
    ax.set_aspect('equal')
    plt.title(title, fontsize=12, pad=15)
    plt.xlabel("Largura")
    plt.ylabel("Altura")
    plt.grid(False)
    plt.show()

In [9]:
def start_interactive_app():
    csv_files = [f for f in os.listdir() if f.endswith('.csv')]

    # Widgets
    file_dropdown = Dropdown(options=csv_files, description='Arquivo CSV:', style={'description_width': 'initial'})
    alpha_slider = FloatSlider(value=0.0, min=0.0, max=1.0, step=0.1, description='Alpha (GASP):', style={'description_width': 'initial'})
    width_input = IntSlider(value=10, min=5, max=50, description='Largura Bin:', style={'description_width': 'initial'})
    height_input = IntSlider(value=20, min=5, max=100, description='Altura Bin:', style={'description_width': 'initial'})
    run_button = Button(description="Gerar Solução", button_style='primary')

    output_area = Output()
    step_output = Output()

    def on_run_clicked(b):
        output_area.clear_output()
        step_output.clear_output()

        with output_area:
            items = load_items_from_csv(file_dropdown.value)
            if not items:
                print("Erro ao carregar itens.")
                return

            history, decisions = shelf_first_fit_gasp(items, width_input.value, height_input.value, alpha_slider.value)

            def show_step(step):
                step_output.clear_output()
                with step_output:
                    print(f"Decisão: {decisions[step]}")
                    plot_bin_step(history[step], title=f"Passo {step} de {len(history)-1}")

            interact(show_step, step=IntSlider(min=0, max=len(history)-1, step=1, value=len(history)-1, description='Passo:'))

    run_button.on_click(on_run_clicked)

    # Layout
    controls = VBox([
        Label(value="Configurações do Problema"),
        HBox([file_dropdown, alpha_slider]),
        HBox([width_input, height_input]),
        run_button
    ])

    display(controls, output_area, step_output)


In [10]:
if __name__ == "__main__":
    start_interactive_app()

    items = load_items_from_csv("exemplo_variado.csv")
    history, decisions = shelf_first_fit_gasp(items, 10, 20, alpha=0.5)
    plt.savefig("final_step.png")

Output()

Output()

<Figure size 640x480 with 0 Axes>